In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, when, round

spark = SparkSession.builder \
    .appName("HoldingCompanyPipeline") \
    .getOrCreate()

spark

In [3]:
telco_df = spark.read.csv(
    "file:///home/bigdata/Desktop/WA_Fn-UseC_-Telco-Customer-Churn.csv",
    header=True,
    inferSchema=True
)

risk_df = spark.read.csv(
    "file:///home/bigdata/Desktop/big4_financial_risk_compliance.csv",
    header=True,
    inferSchema=True
)

In [4]:
telco_df.printSchema()
telco_df.show(5)

risk_df.printSchema()
risk_df.show(5)

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)

+----------+------+-------------+-------+----------+------+------------+---------

In [5]:
telco_df = telco_df.dropDuplicates()

In [6]:
risk_df = risk_df.dropDuplicates()

In [7]:
for field in telco_df.schema.fields:
    if isinstance(field.dataType, StringType):
        telco_df = telco_df.withColumn(field.name, trim(col(field.name)))

In [8]:
telco_df = telco_df.withColumn(
    "TotalCharges",
    when(col("TotalCharges") == "", None)
    .otherwise(col("TotalCharges").cast("double"))
)

telco_df = telco_df.fillna({"TotalCharges": 0.0})

In [9]:
binary_cols_to_clean = [
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for c in binary_cols_to_clean:
    telco_df = telco_df.withColumn(
        c,
        when(
            col(c).isin("No internet service", "No phone service"),
            "No"
        ).otherwise(col(c))
    )

In [10]:
telco_df = telco_df.withColumn(
    "AvgMonthlySpend",
    round(
        col("TotalCharges") /
        when(col("tenure") == 0, 1).otherwise(col("tenure")),
        2
    )
)

telco_df = telco_df.withColumn(
    "TenureYears",
    round(col("tenure") / 12, 1)
)

telco_df = telco_df.withColumn(
    "TenureGroup",
    when(col("tenure") <= 12, "0-1 Year")
    .when(col("tenure") <= 24, "1-2 Years")
    .when(col("tenure") <= 48, "2-4 Years")
    .otherwise("4+ Years")
)


In [11]:
telco_df.cache()

DataFrame[customerID: string, gender: string, SeniorCitizen: int, Partner: string, Dependents: string, tenure: int, PhoneService: string, MultipleLines: string, InternetService: string, OnlineSecurity: string, OnlineBackup: string, DeviceProtection: string, TechSupport: string, StreamingTV: string, StreamingMovies: string, Contract: string, PaperlessBilling: string, PaymentMethod: string, MonthlyCharges: double, TotalCharges: double, Churn: string, AvgMonthlySpend: double, TenureYears: double, TenureGroup: string]

In [12]:
telco_df.show(5)
telco_df.printSchema()

+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+---------------+-----------+-----------+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|AvgMonthlySpend|TenureYears|TenureGroup|
+----------+------+-------------+-------+----------+------+------------+-------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+---------------+-----------+-----------+
|8637-XJIVR|Female|            0|     No|        No|   

In [13]:
for field in risk_df.schema.fields:
    if isinstance(field.dataType, StringType):
        risk_df = risk_df.withColumn(field.name, trim(col(field.name)))

In [14]:
risk_df = risk_df.fillna({
    "High_Risk_Cases": 0,
    "Fraud_Cases_Detected": 0,
    "Total_Revenue_Impact": 0.0,
    "Audit_Effectiveness_Score": 0.0,
    "Client_Satisfaction_Score": 0.0
})

In [15]:
risk_df = risk_df.withColumn(
    "High_Risk_Ratio",
    round(
        col("High_Risk_Cases") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        4
    )
)

In [16]:
risk_df = risk_df.withColumn(
    "Fraud_Detection_Rate",
    round(
        col("Fraud_Cases_Detected") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        4
    )
)

In [17]:
risk_df = risk_df.withColumn(
    "Avg_Revenue_Impact_Per_Engagement",
    round(
        col("Total_Revenue_Impact") /
        when(col("Total_Audit_Engagements") == 0, 1)
        .otherwise(col("Total_Audit_Engagements")),
        2
    )
)

In [18]:
risk_df = risk_df.withColumn(
    "Audit_Quality_Score",
    round(
        (col("Audit_Effectiveness_Score") +
         col("Client_Satisfaction_Score")) / 2,
        2
    )
)

In [19]:
risk_df.cache()


DataFrame[Year: int, Firm_Name: string, Total_Audit_Engagements: int, High_Risk_Cases: int, Compliance_Violations: int, Fraud_Cases_Detected: int, Industry_Affected: string, Total_Revenue_Impact: double, AI_Used_for_Auditing: string, Employee_Workload: int, Audit_Effectiveness_Score: double, Client_Satisfaction_Score: double, High_Risk_Ratio: double, Fraud_Detection_Rate: double, Avg_Revenue_Impact_Per_Engagement: double, Audit_Quality_Score: double]

In [20]:
risk_df.show(5)
risk_df.printSchema()

+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+---------------+--------------------+---------------------------------+-------------------+
|Year|Firm_Name|Total_Audit_Engagements|High_Risk_Cases|Compliance_Violations|Fraud_Cases_Detected|Industry_Affected|Total_Revenue_Impact|AI_Used_for_Auditing|Employee_Workload|Audit_Effectiveness_Score|Client_Satisfaction_Score|High_Risk_Ratio|Fraud_Detection_Rate|Avg_Revenue_Impact_Per_Engagement|Audit_Quality_Score|
+----+---------+-----------------------+---------------+---------------------+--------------------+-----------------+--------------------+--------------------+-----------------+-------------------------+-------------------------+---------------+--------------------+---------------------------------+-------------------+
|2021|     KPMG|                   25

In [22]:
telco_df.createOrReplaceTempView("telco")

In [23]:
#Spark SQL
spark.sql("""
SELECT COUNT(*) AS Total_Customers
FROM telco
""").show()

+---------------+
|Total_Customers|
+---------------+
|           7043|
+---------------+



In [29]:
spark.sql("""
SELECT ROUND(AVG(Churn) * 100, 2) AS Churn_Rate
FROM telco
""").show()

+----------+
|Churn_Rate|
+----------+
|      null|
+----------+



In [30]:
spark.sql("""
SELECT Contract,ROUND(SUM(TotalCharges),2) AS Revenue
FROM telco
GROUP BY Contract
ORDER BY Revenue DESC
""").show()

+--------------+---------+
|      Contract|  Revenue|
+--------------+---------+
|      Two year|6283253.7|
|Month-to-month|5305861.5|
|      One year|4467053.5|
+--------------+---------+



In [31]:
spark.sql("""
SELECT TenureGroup,ROUND(AVG(AvgMonthlySpend),2) AS Avg_Spend
FROM telco
GROUP BY TenureGroup
ORDER BY Avg_Spend DESC
""").show()

+-----------+---------+
|TenureGroup|Avg_Spend|
+-----------+---------+
|   4+ Years|    73.97|
|  2-4 Years|    65.87|
|  1-2 Years|     61.3|
|   0-1 Year|    55.94|
+-----------+---------+



In [32]:
risk_df.createOrReplaceTempView("financial")

In [33]:
spark.sql("""
SELECT ROUND(AVG(High_Risk_Ratio),4) AS Avg_Risk
FROM financial
""").show()

+--------+
|Avg_Risk|
+--------+
|   0.134|
+--------+



In [34]:
spark.sql("""
SELECT Industry_Affected,ROUND(AVG(High_Risk_Ratio),4) AS Risk
FROM financial
GROUP BY Industry_Affected
ORDER BY Risk DESC
""").show()

+-----------------+------+
|Industry_Affected|  Risk|
+-----------------+------+
|          Finance|0.1534|
|       Healthcare|0.1438|
|             Tech| 0.126|
|           Retail|0.1194|
+-----------------+------+



In [35]:
spark.sql("""
SELECT AI_Used_for_Auditing,ROUND(AVG(Audit_Quality_Score),2) AS Avg_Quality
FROM financial
GROUP BY AI_Used_for_Auditing
""").show()

+--------------------+-----------+
|AI_Used_for_Auditing|Avg_Quality|
+--------------------+-----------+
|                  No|       7.36|
|                 Yes|       7.48|
+--------------------+-----------+



In [36]:
spark.sql("""
SELECT Firm_Name,ROUND(SUM(Total_Revenue_Impact),2) AS Revenue_Impact
FROM financial
GROUP BY Firm_Name
ORDER BY Revenue_Impact DESC
LIMIT 10
""").show()

+-------------+--------------+
|    Firm_Name|Revenue_Impact|
+-------------+--------------+
|     Deloitte|       8547.44|
|Ernst & Young|       6862.86|
|          PwC|       6108.49|
|         KPMG|       5735.11|
+-------------+--------------+



In [37]:
telco_df.write.mode("overwrite").parquet("/home/bigdata/Desktop/output/telco")

In [38]:
risk_df.write.mode("overwrite").parquet("/home/bigdata/Desktop/output/risk")